# 03. Model Training & Evaluation

This notebook trains and evaluates multiple machine learning algorithms:
- **Logistic Regression** (Baseline)
- **Random Forest** (Hyperparameter optimization via `RandomizedSearchCV` tuned for PR-AUC)
- **XGBoost Classifier**
- Comprehensive comparison matrix outputted to `reports/model_performance.csv`

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_recall_curve,
    auc,
    precision_score,
    recall_score,
    f1_score
)
from imblearn.over_sampling import SMOTE

sns.set(style="whitegrid")
os.makedirs("../models", exist_ok=True)
os.makedirs("../reports", exist_ok=True)

# Data Pipeline Reconstruction
data = pd.read_csv("../data/creditcard.csv")
df = data.copy()
df["Log_Amount"] = np.log1p(df["Amount"])
df["Is_High_Amount"] = (df["Amount"] >= 100).astype(int)

X = df.drop("Class", axis=1)
y = df["Class"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled, X_test_scaled = X_train.copy(), X_test.copy()
for col in ["Time", "Amount", "Log_Amount"]:
    X_train_scaled[col] = scaler.fit_transform(X_train[[col]])
    X_test_scaled[col] = scaler.transform(X_test[[col]])

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

rf_temp = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_temp.fit(X_train_res, y_train_res)
selector = SelectFromModel(rf_temp, threshold="median", prefit=True)
X_train_sel = selector.transform(X_train_res)
X_test_sel = selector.transform(X_test_scaled)
selected_features = X_train_scaled.columns[selector.get_support()]

## 1. Evaluation Utility Function

In [ ]:
def evaluate_model(name, y_true, y_pred, y_proba):
    print(f"\n=== {name} Report ===")
    print(classification_report(y_true, y_pred, digits=4))

    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(f"Confusion Matrix - {name}")
    plt.show()

    roc = roc_auc_score(y_true, y_proba)
    precision, recall, _ = precision_recall_curve(y_true, y_proba)
    pr_auc = auc(recall, precision)

    print(f"{name} ROC-AUC:", roc)
    print(f"{name} PR-AUC:", pr_auc)

    plt.plot(recall, precision)
    plt.title(f"Precision-Recall Curve - {name}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.show()

    return roc, pr_auc

## 2. Train Models

In [ ]:
results = {}

# Baseline Logistic Regression
log_reg = LogisticRegression(max_iter=1000, n_jobs=-1)
log_reg.fit(X_train_sel, y_train_res)
joblib.dump(log_reg, "../models/logistic_regression.pkl")
y_proba_lr = log_reg.predict_proba(X_test_sel)[:, 1]
y_pred_lr = (y_proba_lr >= 0.5).astype(int)
results["Logistic Regression"] = evaluate_model("Logistic Regression", y_test, y_pred_lr, y_proba_lr)

# Tuned Random Forest
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
param_grid = {
    "n_estimators": [100, 150],
    "max_depth": [10, 20, None],
    "min_samples_split": [2, 5]
}
rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)
search = RandomizedSearchCV(rf_base, param_grid, n_iter=6, scoring="average_precision", cv=cv, random_state=42, n_jobs=-1, verbose=1)
search.fit(X_train_sel, y_train_res)
rf = search.best_estimator_
print(f"Best Hyperparameters: {search.best_params_}")
joblib.dump(rf, "../models/random_forest.pkl")
y_proba_rf = rf.predict_proba(X_test_sel)[:, 1]
y_pred_rf = (y_proba_rf >= 0.5).astype(int)
results["Random Forest"] = evaluate_model("Random Forest", y_test, y_pred_rf, y_proba_rf)

# XGBoost
xgb = None
try:
    from xgboost import XGBClassifier
    xgb = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, objective="binary:logistic", n_jobs=-1, random_state=42)
    xgb.fit(X_train_sel, y_train_res)
    joblib.dump(xgb, "../models/xgboost.pkl")
    y_proba_xgb = xgb.predict_proba(X_test_sel)[:, 1]
    y_pred_xgb = (y_proba_xgb >= 0.5).astype(int)
    results["XGBoost"] = evaluate_model("XGBoost", y_test, y_pred_xgb, y_proba_xgb)
except ImportError:
    print("XGBoost not installed.")

## 3. Multi-Model Performance Comparison

In [ ]:
model_preds = {
    "Logistic Regression": (y_pred_lr, y_proba_lr),
    "Random Forest": (y_pred_rf, y_proba_rf)
}
if xgb is not None:
    model_preds["XGBoost"] = (y_pred_xgb, y_proba_xgb)

comparison_rows = []
perf_rows = []

for name, (preds, probs) in model_preds.items():
    cm = confusion_matrix(y_test, preds)
    roc = results[name][0]
    pr = results[name][1]
    perf_rows.append([name, roc, pr])
    comparison_rows.append({
        "Model": name,
        "Precision": precision_score(y_test, preds),
        "Recall": recall_score(y_test, preds),
        "F1-Score": f1_score(y_test, preds),
        "False Positives (FP)": cm[0, 1],
        "False Negatives (FN)": cm[1, 0],
        "ROC-AUC": roc,
        "PR-AUC": pr
    })

pd.DataFrame(perf_rows, columns=["Model", "ROC-AUC", "PR-AUC"]).to_csv("../reports/model_performance.csv", index=False)
comp_df = pd.DataFrame(comparison_rows)
print("\n=== COMPREHENSIVE MULTI-MODEL COMPARISON MATRIX ===")
print(comp_df.to_string(index=False))